# Restored, code-first edition

This edition restores the original architecture explanations, interactive browser, hands-on NumPy/PyTorch labs, tensor-shape walkthroughs, use-case catalog, and per-architecture code snippets. It expands the neural-network catalog to 45 entries and adds runnable classical ML and broader AI examples. The examples are educational reference implementations, not all production-grade or exact reproductions of large research models.

# Deep Learning & Neural Network Architecture Explorer
### Interactive, Python-first notebook | foundations → architectures → use cases → experiments

This notebook is a practical field guide to **32 neural-network architecture patterns and families**, plus a catalog of **40 representative use cases**. It deliberately stays below the 50-architecture ceiling while covering foundational, common, specialized, and newer approaches.

**Learning path**
1. Understand a neuron, tensors, activations, loss, gradients, and backpropagation.
2. Browse architecture cards and compare their data flow, strengths, trade-offs, and applications.
3. Run a small neural network implemented from scratch in NumPy.
4. Inspect compact PyTorch implementation patterns.
5. Explore the use-case catalog and select an architecture based on the data and task.
6. Review production scaling concepts separately from model architecture.

**Execution:** Core experiments use NumPy and Matplotlib. The architecture browser uses `ipywidgets` when available. PyTorch snippets are reference implementations; they are not all executed automatically. A CPU is enough for the included labs.

> Install dependencies if needed: `pip install numpy matplotlib ipywidgets torch`. If widgets do not render in your Jupyter environment, use the `show_architecture(name)` function directly.


In [ ]:
# 0. Environment and imports
import math
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("NumPy:", np.__version__)
print("Interactive widgets available:", WIDGETS_AVAILABLE)


## 1. The common neural-network building blocks

An architecture is the **arrangement and connectivity of learnable operations**. Training is the process that adjusts parameters. Backpropagation is a way to compute gradients; it is not itself an architecture.

A typical supervised learning loop is:

`input → model → prediction → loss → gradients → optimizer update`

Important dimensions:
- **Batch (`B`)**: number of examples processed together.
- **Features/channels (`F` or `C`)**: values available at each example or location.
- **Sequence length (`T`)**: number of time steps or tokens.
- **Hidden dimension (`D`)**: width of an internal representation.
- **Classes (`K`)**: number of output categories.

Tensor shape is part of the model contract. Always check it at layer boundaries.


In [ ]:
# A tiny neuron: y = activation(x @ W + b)
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

x = np.array([[0.7, 0.2, 0.9]])          # (batch=1, features=3)
W = np.array([[0.4], [-0.3], [0.8]])     # (features=3, output_units=1)
b = np.array([0.1])                      # (output_units=1)

z = x @ W + b                            # (1, 1)
y = sigmoid(z)
print("input shape:", x.shape)
print("weight shape:", W.shape)
print("pre-activation z:", z)
print("sigmoid output:", y)


In [ ]:
# Visualize common activation functions
x = np.linspace(-6, 6, 400)
relu = np.maximum(0, x)
tanh = np.tanh(x)
sig = sigmoid(x)
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

plt.figure(figsize=(9, 4.5))
plt.plot(x, relu, label="ReLU")
plt.plot(x, tanh, label="tanh")
plt.plot(x, sig, label="sigmoid")
plt.plot(x, gelu, label="GELU")
plt.axhline(0, linewidth=0.7)
plt.axvline(0, linewidth=0.7)
plt.title("Activation functions")
plt.xlabel("input")
plt.ylabel("activation")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 2. Architecture catalog

The catalog contains **40 architecture entries**. Some entries are individual architectures; others are widely used architectural families or patterns. They are grouped by the primary structural idea, not by application domain.

Use the interactive browser below to compare the purpose, data flow, strengths, limitations, use cases, and tensor shapes. The code tab shows a compact PyTorch pattern where appropriate.


In [ ]:
ARCHITECTURES = [{'name': 'Perceptron', 'family': 'Foundations', 'idea': 'A single linear unit followed by a threshold/activation. It learns a linear decision boundary.', 'flow': 'Features → weighted sum + bias → activation → output', 'strengths': 'Smallest useful model; makes weights, bias, and decision boundaries concrete.', 'limitations': 'A single perceptron cannot represent non-linearly separable patterns such as XOR.', 'uses': 'Simple binary decisions; teaching the neuron abstraction.', 'code': 'import torch\nfrom torch import nn\n\nclass Perceptron(nn.Module):\n    def __init__(self, n_features):\n        super().__init__()\n        self.linear = nn.Linear(n_features, 1)\n\n    def forward(self, x):\n        return torch.sigmoid(self.linear(x))', 'shapes': 'Input (B,F) → output (B,1).'}, {'name': 'Multilayer Perceptron (MLP)', 'family': 'Feed-forward', 'idea': 'Stacks fully connected layers. Nonlinear activations let it learn nonlinear mappings from fixed-size vectors.', 'flow': 'Feature vector → Dense → activation → Dense → output', 'strengths': 'Strong baseline for tabular data; flexible and easy to debug.', 'limitations': 'Does not automatically exploit image locality, graph structure, or sequence order.', 'uses': 'Churn prediction, fraud scoring, tabular regression, classification heads.', 'code': 'import torch\nfrom torch import nn\n\nclass MLP(nn.Module):\n    def __init__(self, n_features, n_classes):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(n_features, 128), nn.ReLU(),\n            nn.Dropout(0.2),\n            nn.Linear(128, 64), nn.ReLU(),\n            nn.Linear(64, n_classes)\n        )\n\n    def forward(self, x):\n        return self.net(x)  # logits for CrossEntropyLoss', 'shapes': '(B,F) → (B,K); logits are not probabilities.'}, {'name': 'Convolutional Neural Network (CNN)', 'family': 'Convolutional / vision', 'idea': 'Learns local filters that slide across spatial dimensions. Early layers often detect edges/textures; deeper layers combine them into parts and objects.', 'flow': 'Image → convolution → activation → pooling/stride → feature maps → head', 'strengths': 'Weight sharing reduces parameters; strong spatial inductive bias.', 'limitations': 'Large receptive fields/global context may require deeper stacks, dilation, attention, or multi-scale design.', 'uses': 'Image classification, defect detection, medical imaging, audio spectrograms.', 'code': 'import torch\nfrom torch import nn\n\ncnn = nn.Sequential(\n    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),\n    nn.MaxPool2d(2),\n    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),\n    nn.AdaptiveAvgPool2d((1, 1)),\n    nn.Flatten(),\n    nn.Linear(64, 10)\n)\n# Input: (B, 3, H, W); output: (B, 10)', 'shapes': '(B,C,H,W) → feature maps → (B,K).'}, {'name': 'LeNet-style CNN', 'family': 'Convolutional / vision', 'idea': 'An early compact CNN pattern: convolution/pooling stages followed by dense classification layers.', 'flow': 'Small image → Conv/Pool → Conv/Pool → Dense → class', 'strengths': 'Simple historical baseline; useful for learning convolutional feature extraction.', 'limitations': 'Limited capacity and design assumptions compared with modern image models.', 'uses': 'Digit recognition, small grayscale image classification, teaching CNNs.', 'code': 'import torch.nn as nn\n\nlenet = nn.Sequential(\n    nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),\n    nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2),\n    nn.Flatten(),\n    nn.Linear(16 * 4 * 4, 120), nn.Tanh(),\n    nn.Linear(120, 84), nn.Tanh(),\n    nn.Linear(84, 10)\n)\n# Classic sizing assumes 28x28 grayscale input.', 'shapes': '(B,1,28,28) → (B,10), with classic LeNet-5 sizing.'}, {'name': 'VGG-style CNN', 'family': 'Convolutional / vision', 'idea': 'Uses repeated small 3×3 convolutions and pooling in a simple, regular stack.', 'flow': 'Image → [Conv3×3 + ReLU]×N → Pool → deeper blocks → classifier', 'strengths': 'Conceptually simple; repeated blocks make depth easy to reason about.', 'limitations': 'Can be parameter- and compute-heavy, especially with large dense heads.', 'uses': 'Image classification baselines; feature extraction demonstrations.', 'code': 'import torch.nn as nn\n\nvgg_block = nn.Sequential(\n    nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),\n    nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),\n    nn.MaxPool2d(2)\n)', 'shapes': 'Spatial dimensions are halved by each 2×2 pooling stage.'}, {'name': 'ResNet (Residual Network)', 'family': 'Convolutional / vision', 'idea': 'Adds identity skip connections so a block learns a residual update: y = F(x) + x. This improves gradient flow in deep networks.', 'flow': 'x → residual block F(x) → add skip x → activation', 'strengths': 'Enables very deep CNNs; skip paths ease optimization.', 'limitations': 'Shape/channel changes require projection shortcuts; still compute-intensive at scale.', 'uses': 'Image classification, detection backbones, medical vision, transfer learning.', 'code': 'import torch\nfrom torch import nn\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.f = nn.Sequential(\n            nn.Conv2d(channels, channels, 3, padding=1, bias=False),\n            nn.BatchNorm2d(channels), nn.ReLU(),\n            nn.Conv2d(channels, channels, 3, padding=1, bias=False),\n            nn.BatchNorm2d(channels)\n        )\n        self.act = nn.ReLU()\n\n    def forward(self, x):\n        return self.act(x + self.f(x))', 'shapes': 'Input and residual output must have matching shapes for identity addition.'}, {'name': 'DenseNet', 'family': 'Convolutional / vision', 'idea': 'Connects each layer to later layers by concatenating feature maps, encouraging feature reuse.', 'flow': 'x₀ → layer₁; [x₀,x₁] → layer₂; [x₀,x₁,x₂] → …', 'strengths': 'Feature reuse and strong gradient paths; can be parameter-efficient.', 'limitations': 'Concatenation increases activation-memory demands; implementation needs careful channel bookkeeping.', 'uses': 'Image classification, feature extraction, some medical imaging pipelines.', 'code': 'import torch\nfrom torch import nn\n\nclass DenseLayer(nn.Module):\n    def __init__(self, in_channels, growth=16):\n        super().__init__()\n        self.layer = nn.Sequential(\n            nn.BatchNorm2d(in_channels), nn.ReLU(),\n            nn.Conv2d(in_channels, growth, 3, padding=1)\n        )\n\n    def forward(self, x):\n        new_features = self.layer(x)\n        return torch.cat([x, new_features], dim=1)', 'shapes': 'Channels grow by `growth` after each dense layer.'}, {'name': 'EfficientNet / compound-scaled CNN', 'family': 'Convolutional / vision', 'idea': 'Scales network depth, width, and input resolution together rather than increasing only one dimension.', 'flow': 'Efficient convolutional blocks → compound scaling → classifier', 'strengths': 'Balances accuracy and compute across model sizes.', 'limitations': 'Scaling recipe and pretrained variants matter; not every task benefits equally.', 'uses': 'Mobile/edge vision, image classification, efficient feature extraction.', 'code': 'import torch\nfrom torchvision.models import efficientnet_b0\n\nmodel = efficientnet_b0(weights=None)\n# Replace classifier for a custom number of classes:\n# model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, n_classes)', 'shapes': 'Typical torchvision input: (B,3,H,W); classifier output depends on model head.'}, {'name': 'U-Net', 'family': 'Convolutional / vision', 'idea': 'Encoder–decoder CNN with skip connections between matching resolutions. The decoder restores spatial detail for dense predictions.', 'flow': 'Image → downsampling encoder → bottleneck → upsampling decoder + skips → pixel map', 'strengths': 'Excellent spatial localization; multi-scale features combine context and detail.', 'limitations': 'High-resolution feature maps consume memory; 3D and multimodal variants need more compute.', 'uses': 'Medical segmentation, satellite segmentation, defect localization, diffusion backbones.', 'code': 'import torch\nfrom torch import nn\n\nclass ConvBlock(nn.Module):\n    def __init__(self, c_in, c_out):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(),\n            nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU()\n        )\n    def forward(self, x): return self.net(x)\n\n# U-Net pattern: encoder feature maps are concatenated into decoder stages.\n# A complete model also needs downsampling, upsampling, and matching skip shapes.', 'shapes': 'Dense output: (B,classes,H,W), often trained with pixel-wise loss.'}, {'name': 'RNN', 'family': 'Sequence / recurrent', 'idea': 'Processes one time step at a time and carries a hidden state forward.', 'flow': 'x₁ → RNN cell → h₁ → RNN cell → h₂ → …', 'strengths': 'Naturally handles ordered sequences and streaming step-by-step computation.', 'limitations': 'Long-range dependencies can be difficult due to vanishing/exploding gradients; sequential computation limits parallelism.', 'uses': 'Time series, sequence tagging, small sequence baselines.', 'code': 'import torch\nfrom torch import nn\n\nrnn = nn.RNN(input_size=12, hidden_size=64, batch_first=True)\nx = torch.randn(8, 30, 12)       # (B,T,F)\nsequence, h_n = rnn(x)\n# sequence: (8,30,64); h_n: (1,8,64)', 'shapes': '(B,T,F) → (B,T,H), plus final hidden state.'}, {'name': 'LSTM', 'family': 'Sequence / recurrent', 'idea': 'Adds a cell state and forget/input/output gates to control what information is retained, written, and exposed.', 'flow': 'Sequence → gated cell state + hidden state → output', 'strengths': 'More capable than a vanilla RNN at preserving useful long-range information.', 'limitations': 'More gates and parameters; still sequential across time steps.', 'uses': 'Demand forecasting, sensor sequences, speech, event sequences.', 'code': 'import torch\nfrom torch import nn\n\nlstm = nn.LSTM(input_size=12, hidden_size=64, num_layers=2,\n               batch_first=True, bidirectional=False)\nx = torch.randn(8, 30, 12)\nsequence, (h_n, c_n) = lstm(x)\n# sequence: (8,30,64); h_n/c_n: (2,8,64)', 'shapes': 'Input (B,T,F); sequence output (B,T,H*num_directions).'}, {'name': 'GRU', 'family': 'Sequence / recurrent', 'idea': 'Uses update and reset gates to regulate memory, with a simpler state design than LSTM.', 'flow': 'Sequence → update/reset gates → hidden state → output', 'strengths': 'Often a useful lower-complexity recurrent baseline.', 'limitations': 'Still sequential; performance relative to LSTM is task-dependent.', 'uses': 'Sensor analytics, time-series classification, compact sequence models.', 'code': 'import torch\nfrom torch import nn\n\ngru = nn.GRU(input_size=12, hidden_size=64, batch_first=True)\nx = torch.randn(8, 30, 12)\nsequence, h_n = gru(x)', 'shapes': '(B,T,F) → (B,T,H); h_n contains final hidden state.'}, {'name': 'Bidirectional RNN / LSTM', 'family': 'Sequence / recurrent', 'idea': 'Runs recurrent layers in both forward and reverse directions and combines both contextual views.', 'flow': 'Sequence → forward encoder + backward encoder → concatenate states', 'strengths': 'Uses both past and future context for each position.', 'limitations': 'Not naturally causal for real-time streaming; roughly doubles recurrent state width.', 'uses': 'Offline text tagging, named-entity recognition, sequence labeling.', 'code': 'import torch.nn as nn\n\nbilstm = nn.LSTM(input_size=32, hidden_size=64,\n                 batch_first=True, bidirectional=True)\n# Output feature width is 128 because forward and backward states are concatenated.', 'shapes': '(B,T,F) → (B,T,2H).'}, {'name': 'Seq2Seq Encoder–Decoder', 'family': 'Sequence / recurrent', 'idea': 'An encoder reads an input sequence and a decoder generates a target sequence. Attention is commonly added to avoid relying on a single fixed context.', 'flow': 'Source tokens → encoder states → decoder (+ attention) → target tokens', 'strengths': 'Maps sequences of different lengths; general encoder/decoder pattern.', 'limitations': 'A fixed-vector bottleneck can hurt long inputs without attention; decoding may be sequential.', 'uses': 'Translation, summarization, speech recognition, sequence conversion.', 'code': 'import torch\nfrom torch import nn\n\nembedding = nn.Embedding(num_embeddings=5000, embedding_dim=128)\nencoder = nn.LSTM(128, 256, batch_first=True)\ndecoder = nn.LSTM(128, 256, batch_first=True)\n# Encode source tokens, pass state to decoder, and add attention/output projection.', 'shapes': 'Source (B,Tsrc) and target (B,Ttgt) can have different lengths.'}, {'name': 'Transformer Encoder', 'family': 'Attention / Transformer', 'idea': 'Uses multi-head self-attention and feed-forward blocks with residual connections and normalization. Each token can integrate information from other tokens.', 'flow': 'Tokens + positions → self-attention → feed-forward → repeated blocks', 'strengths': 'Parallelizable training; strong global context modeling.', 'limitations': 'Full attention has quadratic pairwise cost in sequence length; positional information is needed.', 'uses': 'Text classification, embeddings, tabular token models, vision encoders.', 'code': 'import torch\nfrom torch import nn\n\nlayer = nn.TransformerEncoderLayer(\n    d_model=128, nhead=4, batch_first=True, norm_first=True\n)\nencoder = nn.TransformerEncoder(layer, num_layers=4)\nx = torch.randn(8, 100, 128)  # (B,T,D)\ny = encoder(x)', 'shapes': '(B,T,D) → (B,T,D).'}, {'name': 'Transformer Decoder / GPT-style', 'family': 'Attention / Transformer', 'idea': 'A decoder-only Transformer uses causal masking so each token can attend only to itself and earlier tokens. It predicts the next token autoregressively.', 'flow': 'Prefix tokens → causal self-attention blocks → next-token logits → append token', 'strengths': 'Scales well for generative sequence modeling; one architecture supports pretraining and many downstream tasks.', 'limitations': 'Autoregressive generation is sequential at inference; long contexts can be expensive.', 'uses': 'LLMs, code generation, text completion, sequence generation.', 'code': 'import torch\nfrom torch import nn\n\n# A causal decoder block can be built with TransformerEncoderLayer\n# plus a causal attention mask, or with a dedicated decoder-only model.\nblock = nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True)\nx = torch.randn(2, 20, 128)\ncausal_mask = torch.triu(torch.ones(20, 20, dtype=torch.bool), diagonal=1)\ny = block(x, src_mask=causal_mask)', 'shapes': 'Causal mask prevents position i from using future positions j > i.'}, {'name': 'Transformer Encoder–Decoder (T5-style pattern)', 'family': 'Attention / Transformer', 'idea': 'An encoder builds contextual source representations; a decoder uses causal self-attention and cross-attention to generate target tokens.', 'flow': 'Source → encoder → memory; target prefix → decoder self-attention + cross-attention → output', 'strengths': 'Flexible conditional generation; explicit source-to-target modeling.', 'limitations': 'More components and memory than decoder-only designs; decoding remains autoregressive.', 'uses': 'Translation, summarization, instruction-to-output transformations.', 'code': 'from torch import nn\n\nmodel = nn.Transformer(\n    d_model=128, nhead=4, num_encoder_layers=3,\n    num_decoder_layers=3, batch_first=True\n)\n# Production use also requires embeddings, positional encodings,\n# source/target masks, and task-specific output projection.', 'shapes': 'Encoder memory (B,S,D); decoder output (B,T,D).'}, {'name': 'Vision Transformer (ViT)', 'family': 'Attention / vision', 'idea': 'Splits an image into patches, projects each patch into a token, adds positional embeddings, then processes tokens with a Transformer encoder.', 'flow': 'Image → patches → patch embeddings + positions → Transformer → class/patch head', 'strengths': 'Global interactions across image patches; unified token-based design.', 'limitations': 'Patch count can be large; data and compute demands may exceed CNN baselines.', 'uses': 'Image classification, visual representation learning, multimodal encoders.', 'code': 'import torch\nfrom torch import nn\n\n# Patch projection via a strided convolution\npatch_embed = nn.Conv2d(3, 192, kernel_size=16, stride=16)\nimage = torch.randn(4, 3, 224, 224)\npatches = patch_embed(image)                  # (4,192,14,14)\ntokens = patches.flatten(2).transpose(1, 2)  # (4,196,192)', 'shapes': '224×224 image with 16×16 patches gives 196 patch tokens.'}, {'name': 'Swin Transformer', 'family': 'Attention / vision', 'idea': 'Uses local window attention and shifted windows to build hierarchical image features with reduced attention cost.', 'flow': 'Patch tokens → window attention → shifted-window blocks → pyramid features', 'strengths': 'More scalable to high-resolution vision than global attention at every layer.', 'limitations': 'Windowing and hierarchical stages add design complexity.', 'uses': 'Detection, segmentation, classification, high-resolution vision.', 'code': '# torchvision provides pretrained Swin variants\nfrom torchvision.models import swin_t\nmodel = swin_t(weights=None)\n# Replace model.head for the target class count.', 'shapes': 'Hierarchical features at progressively lower spatial resolutions.'}, {'name': 'Graph Convolutional Network (GCN)', 'family': 'Graph neural networks', 'idea': 'Updates each node by aggregating transformed features from its neighbors, usually with normalized adjacency.', 'flow': 'Node features + graph edges → neighborhood aggregation → node embeddings', 'strengths': 'Uses graph connectivity directly rather than flattening relationships.', 'limitations': 'Basic GCN smoothing can blur distinctions across many layers; graph batching is specialized.', 'uses': 'Citation networks, fraud rings, molecule properties, node classification.', 'code': '# PyTorch Geometric-style pseudocode\nimport torch\nfrom torch import nn\nfrom torch_geometric.nn import GCNConv\n\nclass GCN(nn.Module):\n    def __init__(self, in_dim, hidden, out_dim):\n        super().__init__()\n        self.conv1 = GCNConv(in_dim, hidden)\n        self.conv2 = GCNConv(hidden, out_dim)\n    def forward(self, x, edge_index):\n        x = self.conv1(x, edge_index).relu()\n        return self.conv2(x, edge_index)', 'shapes': 'Node features (N,F), edges as edge_index (2,E) → node logits (N,K).'}, {'name': 'Graph Attention Network (GAT)', 'family': 'Graph neural networks', 'idea': "Learns attention weights over a node's neighbors, allowing the model to weight neighbor messages differently.", 'flow': 'Node + neighbors → learned neighbor attention → weighted aggregation', 'strengths': 'Adaptive neighbor importance; useful when graph neighborhoods are heterogeneous.', 'limitations': 'Attention over large neighborhoods can be costly; depends on graph quality.', 'uses': 'Recommendation, molecule graphs, social/network analysis, knowledge graphs.', 'code': 'from torch_geometric.nn import GATConv\nfrom torch import nn\n\nclass GAT(nn.Module):\n    def __init__(self, in_dim, hidden, classes):\n        super().__init__()\n        self.gat1 = GATConv(in_dim, hidden, heads=4)\n        self.gat2 = GATConv(hidden * 4, classes, heads=1, concat=False)\n    def forward(self, x, edge_index):\n        return self.gat2(self.gat1(x, edge_index).elu(), edge_index)', 'shapes': 'Output is typically one embedding/logit vector per node.'}, {'name': 'Autoencoder', 'family': 'Latent representation / generative', 'idea': 'Learns to encode an input into a compact latent vector and decode it back. Training minimizes reconstruction error.', 'flow': 'Input → encoder → latent z → decoder → reconstruction', 'strengths': 'Useful for compression, denoising, representation learning, and anomaly scores.', 'limitations': 'Reconstruction quality does not automatically mean the latent space is semantically useful.', 'uses': 'Anomaly detection, denoising, compression, feature learning.', 'code': 'import torch\nfrom torch import nn\n\nclass Autoencoder(nn.Module):\n    def __init__(self, input_dim, latent_dim=16):\n        super().__init__()\n        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(),\n                                     nn.Linear(64, latent_dim))\n        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(),\n                                     nn.Linear(64, input_dim))\n    def forward(self, x):\n        z = self.encoder(x)\n        return self.decoder(z), z', 'shapes': 'Input and reconstruction share shape (B,F); latent is (B,Z).'}, {'name': 'Variational Autoencoder (VAE)', 'family': 'Latent representation / generative', 'idea': 'Learns a probability distribution in latent space (often mean and log-variance), samples z with the reparameterization trick, and reconstructs inputs.', 'flow': 'Input → encoder (μ, log σ²) → sample z → decoder → reconstruction', 'strengths': 'Structured latent space supports sampling and interpolation.', 'limitations': 'Reconstructions may be blurry for images; balancing reconstruction and KL terms matters.', 'uses': 'Generative modeling, anomaly detection, latent interpolation, representation learning.', 'code': 'import torch\nfrom torch import nn\n\nclass VAE(nn.Module):\n    def __init__(self, input_dim, latent_dim=8):\n        super().__init__()\n        self.enc = nn.Linear(input_dim, 64)\n        self.mu = nn.Linear(64, latent_dim)\n        self.logvar = nn.Linear(64, latent_dim)\n        self.dec = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(),\n                                 nn.Linear(64, input_dim))\n    def forward(self, x):\n        h = self.enc(x).relu()\n        mu, logvar = self.mu(h), self.logvar(h)\n        std = torch.exp(0.5 * logvar)\n        z = mu + torch.randn_like(std) * std\n        return self.dec(z), mu, logvar', 'shapes': 'VAE loss = reconstruction term + weighted KL divergence.'}, {'name': 'Generative Adversarial Network (GAN)', 'family': 'Latent representation / generative', 'idea': 'Trains a generator to create samples and a discriminator to distinguish generated from real data. The two networks compete.', 'flow': 'Noise → generator → fake sample; real/fake → discriminator → adversarial losses', 'strengths': 'Can generate sharp samples; flexible generator architectures.', 'limitations': 'Training can be unstable; mode collapse and evaluation are challenging.', 'uses': 'Image synthesis, data augmentation research, image-to-image translation variants.', 'code': 'import torch\nfrom torch import nn\n\ngenerator = nn.Sequential(\n    nn.Linear(100, 256), nn.ReLU(),\n    nn.Linear(256, 784), nn.Tanh()\n)\ndiscriminator = nn.Sequential(\n    nn.Linear(784, 256), nn.LeakyReLU(0.2),\n    nn.Linear(256, 1)\n)\n# Train discriminator on real/fake, then train generator to fool it.', 'shapes': 'Noise (B,100) → generated flattened image (B,784).'}, {'name': 'Diffusion model (U-Net denoiser)', 'family': 'Generative / diffusion', 'idea': 'Learns to reverse a gradual noising process. A denoising network predicts noise or another parameterization at a chosen time step.', 'flow': 'Clean sample → add noise at t; noisy sample + t → denoiser → noise estimate → iterative denoising', 'strengths': 'High-quality controllable generation; stable training objectives.', 'limitations': 'Sampling can require many denoising steps; compute and memory can be substantial.', 'uses': 'Image, audio, video, and scientific-data generation.', 'code': 'import torch\nfrom torch import nn\n\n# Conceptual denoiser: a real image diffusion U-Net is multi-resolution\n# and injects timestep embeddings into residual blocks.\nclass TinyDenoiser(nn.Module):\n    def __init__(self, channels=3):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, 64, 3, padding=1), nn.SiLU(),\n            nn.Conv2d(64, 64, 3, padding=1), nn.SiLU(),\n            nn.Conv2d(64, channels, 3, padding=1)\n        )\n    def forward(self, noisy_x, timestep_embedding=None):\n        return self.net(noisy_x)', 'shapes': 'Denoiser maps noisy sample x_t to predicted noise/velocity/sample.'}, {'name': 'Diffusion Transformer (DiT)', 'family': 'Generative / diffusion', 'idea': 'Uses Transformer blocks over image/video latent patches as the denoising backbone rather than relying only on a convolutional U-Net.', 'flow': 'Noisy latent patches + timestep/conditioning → Transformer blocks → denoising prediction', 'strengths': 'Scales with Transformer design and can model global interactions.', 'limitations': 'Attention and token count can make high-resolution generation expensive.', 'uses': 'High-quality image/video generation research and scalable diffusion backbones.', 'code': 'import torch\nfrom torch import nn\n\n# Architectural sketch: patch tokens, conditioning embeddings, Transformer,\n# and a projection back to patch-shaped denoising targets.\nblocks = nn.TransformerEncoder(\n    nn.TransformerEncoderLayer(256, 8, batch_first=True),\n    num_layers=6\n)\nlatent_tokens = torch.randn(2, 256, 256)  # (B, tokens, hidden)\ndenoising_features = blocks(latent_tokens)', 'shapes': 'Latent patches (B,N,D) → transformed tokens (B,N,D).'}, {'name': 'Siamese Network', 'family': 'Metric learning', 'idea': 'Uses two or more weight-sharing branches to map inputs into a common embedding space; a distance or similarity objective trains the representation.', 'flow': 'Input A → shared encoder → zA; Input B → same encoder → zB; compare', 'strengths': 'Learns similarity rather than only fixed class labels; supports few-shot matching.', 'limitations': 'Needs meaningful positive/negative pairs or triplets; sampling strategy matters.', 'uses': 'Face/product matching, duplicate detection, signature verification, retrieval.', 'code': 'import torch\nfrom torch import nn\n\nclass SiameseEncoder(nn.Module):\n    def __init__(self, in_dim, emb_dim=64):\n        super().__init__()\n        self.encoder = nn.Sequential(nn.Linear(in_dim, 128), nn.ReLU(),\n                                     nn.Linear(128, emb_dim))\n    def forward(self, a, b):\n        return self.encoder(a), self.encoder(b)', 'shapes': 'Two inputs (B,F) → two embeddings (B,D).'}, {'name': 'Contrastive / self-supervised encoder', 'family': 'Metric learning', 'idea': 'Learns representations by bringing related views/examples together and separating unrelated examples, often without manual labels.', 'flow': 'Raw sample → augmented views → shared encoder → embeddings → contrastive loss', 'strengths': 'Uses unlabeled data; pretrained representations transfer to downstream tasks.', 'limitations': 'Augmentation design, negative sampling, and large-batch effects are important.', 'uses': 'Image/text representation learning, retrieval, pretraining, anomaly detection.', 'code': 'import torch\nfrom torch import nn\n\nencoder = nn.Sequential(nn.Linear(512, 256), nn.ReLU(),\n                        nn.Linear(256, 128))\nview_a = torch.randn(32, 512)\nview_b = torch.randn(32, 512)\nz_a = nn.functional.normalize(encoder(view_a), dim=1)\nz_b = nn.functional.normalize(encoder(view_b), dim=1)\n# A contrastive objective (e.g. InfoNCE) compares paired and non-paired embeddings.', 'shapes': 'Each view → normalized embedding (B,D).'}, {'name': 'Deep Q-Network (DQN)', 'family': 'Reinforcement learning', 'idea': 'Uses a neural network to approximate action-value Q(s,a). Experience replay and a target network stabilize learning.', 'flow': 'State → Q-network → action; transition replay → Bellman target → update', 'strengths': 'Combines deep representation learning with value-based reinforcement learning.', 'limitations': 'Primarily suited to discrete action spaces; training can be sample-inefficient and unstable.', 'uses': 'Game playing, discrete control, resource allocation prototypes.', 'code': 'import torch\nfrom torch import nn\n\nq_network = nn.Sequential(\n    nn.Linear(state_dim, 128), nn.ReLU(),\n    nn.Linear(128, 128), nn.ReLU(),\n    nn.Linear(128, n_actions)\n)\n# Select action with epsilon-greedy policy; train selected Q value\n# toward r + gamma * max_a Q_target(next_state, a).', 'shapes': 'State batch (B,S) → Q values (B, number_of_actions).'}, {'name': 'Self-Organizing Map (SOM)', 'family': 'Unsupervised neural maps', 'idea': 'A competitive-learning network maps high-dimensional inputs onto a low-dimensional grid while preserving neighborhood relationships.', 'flow': 'Input → best matching unit → update winner and neighboring prototypes', 'strengths': 'Useful for visual exploration and clustering-like topology maps.', 'limitations': 'Not a modern deep feature learner by default; map size and learning schedule matter.', 'uses': 'Exploratory clustering, customer segmentation visualization, anomaly exploration.', 'code': '# Minimal SOM-style update sketch (NumPy)\n# Find best matching unit (BMU), then move nearby prototype vectors toward x.\ndistances = np.linalg.norm(prototypes - x[None, :], axis=1)\nbmu_index = distances.argmin()\nprototypes[bmu_index] += learning_rate * (x - prototypes[bmu_index])', 'shapes': 'Prototype grid vectors have the same feature width as the input.'}, {'name': 'Restricted Boltzmann Machine (RBM)', 'family': 'Energy-based models', 'idea': 'A stochastic energy-based model with visible and hidden units. It learns a joint distribution using alternating Gibbs sampling, often with contrastive divergence.', 'flow': 'Visible units ↔ hidden units; energy defines probability of configurations', 'strengths': 'Historically important unsupervised representation learner; introduces energy-based modeling.', 'limitations': 'Training and likelihood estimation are less straightforward than common modern models.', 'uses': 'Historical deep learning, feature learning, collaborative filtering research.', 'code': '# Conceptual RBM components; full training requires sampling and contrastive divergence.\nimport torch\nimport torch.nn.functional as F\n\nW = torch.randn(n_visible, n_hidden) * 0.01\nvisible_bias = torch.zeros(n_visible)\nhidden_bias = torch.zeros(n_hidden)\n\ndef hidden_prob(v):\n    return torch.sigmoid(v @ W + hidden_bias)', 'shapes': 'Visible (B,V) ↔ hidden (B,H).'}, {'name': 'Deep Belief Network (DBN)', 'family': 'Energy-based models', 'idea': 'A stack of RBMs trained layer-wise to build hierarchical representations, followed historically by optional supervised fine-tuning.', 'flow': 'Input → RBM layer 1 → RBM layer 2 → … → classifier', 'strengths': 'Historically significant approach to unsupervised pretraining.', 'limitations': 'Largely superseded in many settings by modern end-to-end training and self-supervised methods.', 'uses': 'Historical study of representation learning and deep generative models.', 'code': '# DBNs are commonly taught as stacked RBMs.\n# Modern practice usually favors autoencoders, contrastive learning, or Transformers.', 'shapes': 'Each layer transforms one representation into a higher-level representation.'}, {'name': 'Neural Ordinary Differential Equation (Neural ODE)', 'family': 'Continuous-depth / emerging', 'idea': 'Treats hidden-state evolution as a learned differential equation and uses an ODE solver to compute the output.', 'flow': 'Input → initial state → learned dynamics dz/dt=f(z,t) → ODE solver → output', 'strengths': 'Continuous-depth formulation; adaptive solvers can trade compute for accuracy.', 'limitations': 'Solver cost, numerical stability, and adjoint-gradient details add complexity.', 'uses': 'Irregular time series, continuous dynamics, scientific machine learning.', 'code': '# torchdiffeq is an optional package: pip install torchdiffeq\nimport torch\nfrom torch import nn\nfrom torchdiffeq import odeint\n\nclass Dynamics(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(dim, 64), nn.Tanh(),\n                                 nn.Linear(64, dim))\n    def forward(self, t, z):\n        return self.net(z)\n# trajectory = odeint(Dynamics(dim), z0, time_points)', 'shapes': 'State z(t) has fixed feature width while time is continuous.'}, {'name': 'State Space Model (SSM)', 'family': 'State-space / long sequence', 'idea': 'Models sequences through a latent state updated over time. Modern structured SSMs parameterize transitions to capture long dependencies efficiently.', 'flow': 'Input sequence → state transition/update → output sequence', 'strengths': 'Can scale efficiently with sequence length; useful for long-context signals.', 'limitations': 'Implementation details and hardware kernels matter; not every SSM is interchangeable.', 'uses': 'Long time series, audio, language sequence modeling, scientific signals.', 'code': '# Conceptual discrete state-space recurrence\n# h_t = A h_(t-1) + B x_t; y_t = C h_t + D x_t\n# Modern SSM layers use structured parameterizations and optimized algorithms.', 'shapes': 'Sequence input (B,T,F) → sequence output (B,T,H or F).'}, {'name': 'Mamba / Selective SSM', 'family': 'State-space / long sequence', 'idea': 'A selective state-space architecture makes parts of the state update input-dependent, allowing the model to decide what to retain or forget while maintaining efficient sequence processing.', 'flow': 'Tokens → input-dependent selection/gating → selective state updates → output', 'strengths': 'Designed for efficient long-sequence modeling with linear sequence-length scaling in core selective scan.', 'limitations': 'Specialized kernels and implementation choices affect practical speed; quality is task-dependent and it is not a universal Transformer replacement.', 'uses': 'Long-context language, audio, time series, vision variants, genomics research.', 'code': '# Architecture concept only; use an implementation package/model implementation\n# for actual Mamba layers. The selective scan is a specialized operation.\n# A generic RNN cell is NOT an equivalent implementation.', 'shapes': 'Sequence in (B,T,D) → sequence representation out (B,T,D).'}, {'name': 'Mixture of Experts (MoE)', 'family': 'Sparse / conditional computation', 'idea': 'A router selects a small subset of expert subnetworks for each token or example. Sparse activation increases total parameter capacity without running every expert each time.', 'flow': 'Input token → router/top-k experts → selected expert outputs → combine', 'strengths': 'Large capacity with conditional compute; experts may specialize.', 'limitations': 'Routing balance, communication, memory, and distributed training complicate systems.', 'uses': 'Large language models, scalable multimodal models, conditional computation.', 'code': 'import torch\nfrom torch import nn\n\nclass SparseMoEBlock(nn.Module):\n    def __init__(self, d_model, n_experts=4):\n        super().__init__()\n        self.router = nn.Linear(d_model, n_experts)\n        self.experts = nn.ModuleList([\n            nn.Sequential(nn.Linear(d_model, 4*d_model), nn.GELU(),\n                          nn.Linear(4*d_model, d_model))\n            for _ in range(n_experts)\n        ])\n    def forward(self, x):\n        # Dense illustrative routing; production MoE uses top-k dispatch.\n        weights = self.router(x).softmax(dim=-1)\n        outputs = torch.stack([expert(x) for expert in self.experts], dim=-2)\n        return (outputs * weights.unsqueeze(-1)).sum(dim=-2)', 'shapes': 'Illustrative dense mixture: (B,T,D) → (B,T,D); production top-k is sparse.'}, {'name': 'Dual-encoder multimodal model (CLIP-style)', 'family': 'Multimodal', 'idea': 'Separate encoders map different modalities (e.g. image and text) into a shared embedding space. A contrastive objective aligns matching pairs.', 'flow': 'Image → image encoder → zᵢ; text → text encoder → zₜ; compare similarities', 'strengths': 'Enables cross-modal retrieval and zero-shot transfer through text prompts.', 'limitations': 'Alignment depends on paired data; modality gaps and domain shift remain.', 'uses': 'Image-text search, zero-shot classification, visual search, multimodal retrieval.', 'code': 'import torch\nfrom torch import nn\n\nimage_encoder = nn.Sequential(nn.Flatten(), nn.Linear(3*32*32, 256))\ntext_encoder = nn.Embedding(10000, 256)\n# Pool text token embeddings, project both modalities, normalize,\n# then train matching image/text pairs with a contrastive loss.', 'shapes': 'Image and text embeddings share dimension D for cosine similarity.'}, {'name': 'Perceiver / Perceiver IO', 'family': 'Attention / multimodal', 'idea': 'Uses a small learned latent array that cross-attends to large inputs, then performs processing in latent space to reduce dependence on input size.', 'flow': 'Large input → latent cross-attention → latent processing → output queries', 'strengths': 'Handles large and mixed-modality inputs through a fixed-size latent bottleneck.', 'limitations': 'Latent bottleneck and cross-attention design can constrain information; more specialized than standard Transformer.', 'uses': 'Multimodal perception, video/audio/image inputs, structured outputs.', 'code': 'import torch\nfrom torch import nn\n\ncross_attention = nn.MultiheadAttention(embed_dim=256, num_heads=8, batch_first=True)\nlatents = torch.randn(2, 64, 256)       # fixed latent array\ninputs = torch.randn(2, 4096, 256)     # large input token set\nlatent_updates, _ = cross_attention(latents, inputs, inputs)', 'shapes': 'Input tokens (B,N,D); latent array (B,L,D), where L can be much smaller than N.'}, {'name': 'Neural Radiance Field (NeRF)', 'family': '3D / neural rendering', 'idea': 'Represents a scene as a neural function that maps 3D position and viewing direction to density and color; volume rendering produces images.', 'flow': '3D points + view direction → MLP field → density/color → volume rendering', 'strengths': 'Learns continuous 3D scene representations from images.', 'limitations': 'Original methods can be slow to train/render; capture coverage and scene assumptions matter.', 'uses': 'Novel-view synthesis, 3D reconstruction, robotics/environment modeling.', 'code': 'import torch\nfrom torch import nn\n\nradiance_field = nn.Sequential(\n    nn.Linear(3 + 3, 128), nn.ReLU(),\n    nn.Linear(128, 128), nn.ReLU(),\n    nn.Linear(128, 4)  # RGB-like values + density (conceptual)\n)\n# Inputs combine 3D position and viewing direction.', 'shapes': 'Coordinates (N,3) + direction (N,3) → color and density (N,4).'}, {'name': 'Kolmogorov–Arnold Network (KAN)', 'family': 'Emerging / alternative parameterization', 'idea': 'Uses learnable univariate functions (often spline parameterizations) on connections rather than only fixed scalar weights with node activations.', 'flow': 'Input variables → learnable edge functions → sum at nodes → output', 'strengths': 'Can offer interpretable function components in some settings; active research area.', 'limitations': 'Training cost, scaling, and empirical advantages over MLPs remain task-dependent.', 'uses': 'Scientific function approximation, symbolic/interpretable modeling research.', 'code': '# KAN implementations are provided by third-party research libraries.\n# Conceptually, each edge applies a learnable function phi_ij(x_i),\n# and each destination node sums incoming edge-function values.', 'shapes': 'Input (B,F) → output (B,K); internal edge functions are learnable.'}]

ARCH_BY_NAME = {item['name']: item for item in ARCHITECTURES}
print(f'Loaded {len(ARCHITECTURES)} architecture entries.')

In [ ]:
RUNNABLE_PATCHES = {"Deep Belief Network (DBN)":"import torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nclass BernoulliRBM(nn.Module):\n    def __init__(self, n_visible, n_hidden):\n        super().__init__()\n        self.W = nn.Parameter(torch.randn(n_visible, n_hidden) * 0.01)\n        self.vb = nn.Parameter(torch.zeros(n_visible))\n        self.hb = nn.Parameter(torch.zeros(n_hidden))\n    def sample_h(self, v):\n        p = torch.sigmoid(v @ self.W + self.hb)\n        return p, torch.bernoulli(p)\n    def sample_v(self, h):\n        p = torch.sigmoid(h @ self.W.T + self.vb)\n        return p, torch.bernoulli(p)\n    def contrastive_divergence(self, v0, lr=0.05, steps=1):\n        ph0, h = self.sample_h(v0)\n        for _ in range(steps):\n            pv, v = self.sample_v(h)\n            ph, h = self.sample_h(v)\n        with torch.no_grad():\n            self.W += lr * (v0.T @ ph0 - v.T @ ph) / v0.size(0)\n            self.vb += lr * (v0 - v).mean(0)\n            self.hb += lr * (ph0 - ph).mean(0)\n        return F.binary_cross_entropy(pv, v0)\n\nclass TinyDBN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.rbm1 = BernoulliRBM(20, 12)\n        self.rbm2 = BernoulliRBM(12, 6)\n    def forward(self, x):\n        h1, _ = self.rbm1.sample_h(x)\n        h2, _ = self.rbm2.sample_h(h1)\n        return h2\n\nx = torch.bernoulli(torch.rand(16, 20))\ndbn = TinyDBN()\nloss1 = dbn.rbm1.contrastive_divergence(x)\nh1, _ = dbn.rbm1.sample_h(x)\nloss2 = dbn.rbm2.contrastive_divergence(h1)\nprint('DBN representation:', dbn(x).shape, 'RBM losses:', float(loss1), float(loss2))","Seq2Seq Encoder–Decoder":"import torch\nfrom torch import nn\n\nclass Seq2Seq(nn.Module):\n    def __init__(self, vocab=1000, width=32, hidden=48):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab, width)\n        self.encoder = nn.LSTM(width, hidden, batch_first=True)\n        self.decoder = nn.LSTM(width, hidden, batch_first=True)\n        self.output = nn.Linear(hidden, vocab)\n    def forward(self, source, target_prefix):\n        _, state = self.encoder(self.embedding(source))\n        decoded, _ = self.decoder(self.embedding(target_prefix), state)\n        return self.output(decoded)\n\nsource = torch.randint(0, 1000, (2, 7))\ntarget_prefix = torch.randint(0, 1000, (2, 5))\nlogits = Seq2Seq()(source, target_prefix)\nprint('Seq2Seq target logits:', logits.shape)","Transformer Decoder / GPT-style":"import torch\nfrom torch import nn\n\nvocab, width, length = 500, 64, 10\ntoken_embedding = nn.Embedding(vocab, width)\nposition_embedding = nn.Embedding(length, width)\nblock = nn.TransformerEncoderLayer(width, 4, batch_first=True)\ndecoder = nn.TransformerEncoder(block, num_layers=2)\nhead = nn.Linear(width, vocab)\ntokens = torch.randint(0, vocab, (2, length))\npos = torch.arange(length).unsqueeze(0)\nx = token_embedding(tokens) + position_embedding(pos)\ncausal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)\nlogits = head(decoder(x, mask=causal_mask))\nprint('Causal next-token logits:', logits.shape)","Transformer Encoder–Decoder (T5-style pattern)":"import torch\nfrom torch import nn\n\nmodel = nn.Transformer(d_model=64, nhead=4, num_encoder_layers=2,\n                       num_decoder_layers=2, batch_first=True)\nsource = torch.randn(2, 7, 64)\ntarget = torch.randn(2, 5, 64)\ntgt_mask = nn.Transformer.generate_square_subsequent_mask(5)\noutput = model(source, target, tgt_mask=tgt_mask)\nprint('Encoder-decoder output:', output.shape)","Deep Q-Network (DQN)":"import random\nfrom collections import deque\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nstate_dim, n_actions = 4, 2\nq = nn.Sequential(nn.Linear(state_dim, 32), nn.ReLU(), nn.Linear(32, n_actions))\ntarget = nn.Sequential(nn.Linear(state_dim, 32), nn.ReLU(), nn.Linear(32, n_actions))\ntarget.load_state_dict(q.state_dict())\noptimizer = torch.optim.Adam(q.parameters(), lr=1e-3)\nreplay = deque(maxlen=1000)\nfor _ in range(64):\n    s = torch.randn(state_dim); a = random.randrange(n_actions)\n    r = random.random(); ns = torch.randn(state_dim); done = random.choice([0.0, 1.0])\n    replay.append((s, a, r, ns, done))\nbatch = random.sample(replay, 16)\nstates = torch.stack([b[0] for b in batch])\nactions = torch.tensor([b[1] for b in batch])\nrewards = torch.tensor([b[2] for b in batch])\nnext_states = torch.stack([b[3] for b in batch])\ndones = torch.tensor([b[4] for b in batch])\npred = q(states).gather(1, actions[:, None]).squeeze(1)\nwith torch.no_grad():\n    td_target = rewards + 0.99 * (1 - dones) * target(next_states).max(1).values\nloss = F.smooth_l1_loss(pred, td_target)\noptimizer.zero_grad(); loss.backward(); optimizer.step()\nprint('DQN Bellman loss:', float(loss))","Self-Organizing Map (SOM)":"import numpy as np\n\nrng = np.random.default_rng(42)\ndata = rng.normal(size=(100, 4))\ngrid_h, grid_w = 4, 4\nprototypes = rng.normal(size=(grid_h, grid_w, data.shape[1]))\nfor epoch in range(10):\n    rate = 0.3 * (1 - epoch / 10)\n    radius = max(0.5, 2 * (1 - epoch / 10))\n    for x in data:\n        distances = np.linalg.norm(prototypes - x, axis=2)\n        winner = np.unravel_index(np.argmin(distances), distances.shape)\n        for i in range(grid_h):\n            for j in range(grid_w):\n                d2 = (i-winner[0])**2 + (j-winner[1])**2\n                influence = np.exp(-d2 / (2 * radius**2))\n                prototypes[i, j] += rate * influence * (x - prototypes[i, j])\nprint('SOM prototype grid:', prototypes.shape)","Restricted Boltzmann Machine (RBM)":"import torch\nfrom torch import nn\n\nclass RBM(nn.Module):\n    def __init__(self, visible=20, hidden=10):\n        super().__init__()\n        self.W = nn.Parameter(torch.randn(visible, hidden) * 0.01)\n        self.vb = nn.Parameter(torch.zeros(visible))\n        self.hb = nn.Parameter(torch.zeros(hidden))\n    def hidden(self, v):\n        p = torch.sigmoid(v @ self.W + self.hb)\n        return p, torch.bernoulli(p)\n    def visible(self, h):\n        p = torch.sigmoid(h @ self.W.T + self.vb)\n        return p, torch.bernoulli(p)\n    def cd_step(self, v0, lr=0.05):\n        ph0, h0 = self.hidden(v0)\n        pv1, v1 = self.visible(h0)\n        ph1, _ = self.hidden(v1)\n        with torch.no_grad():\n            self.W += lr * (v0.T @ ph0 - v1.T @ ph1) / v0.size(0)\n            self.vb += lr * (v0 - v1).mean(0)\n            self.hb += lr * (ph0 - ph1).mean(0)\n        return (v0 - pv1).pow(2).mean()\n\nrbm = RBM()\nvisible = torch.bernoulli(torch.rand(16, 20))\nprint('RBM CD reconstruction error:', float(rbm.cd_step(visible)))","Diffusion model (U-Net denoiser)":"import torch\nfrom torch import nn\n\nclass TinyDenoiser(nn.Module):\n    def __init__(self, channels=3, time_width=32):\n        super().__init__()\n        self.time = nn.Sequential(nn.Linear(1, time_width), nn.SiLU(), nn.Linear(time_width, time_width))\n        self.in_conv = nn.Conv2d(channels, 32, 3, padding=1)\n        self.mid = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.SiLU())\n        self.out = nn.Conv2d(32, channels, 3, padding=1)\n        self.time_proj = nn.Linear(time_width, 32)\n    def forward(self, x, t):\n        emb = self.time(t.float().view(-1, 1) / 1000).view(x.size(0), 32, 1, 1)\n        h = self.in_conv(x) + emb\n        return self.out(self.mid(h))\n\ndenoiser = TinyDenoiser()\nclean = torch.randn(4, 3, 16, 16)\nt = torch.randint(0, 1000, (4,))\nnoisy = clean + 0.2 * torch.randn_like(clean)\npredicted_noise = denoiser(noisy, t)\nloss = (predicted_noise - (noisy - clean)).pow(2).mean()\nprint('Denoiser output/loss:', predicted_noise.shape, float(loss))","Diffusion Transformer (DiT)":"import torch\nfrom torch import nn\n\nclass TinyDiT(nn.Module):\n    def __init__(self, width=64, patches=16):\n        super().__init__()\n        self.time = nn.Linear(1, width)\n        self.blocks = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(width, 4, batch_first=True), num_layers=2)\n        self.head = nn.Linear(width, width)\n    def forward(self, noisy_tokens, t):\n        cond = self.time(t.float().view(-1, 1)).unsqueeze(1)\n        return self.head(self.blocks(noisy_tokens + cond))\n\ntokens = torch.randn(2, 16, 64)\nprediction = TinyDiT()(tokens, torch.tensor([10, 20]))\nprint('DiT denoising-token prediction:', prediction.shape)","Dual-encoder multimodal model (CLIP-style)":"import torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nimage_encoder = nn.Sequential(nn.Flatten(), nn.Linear(3*16*16, 64))\ntext_embedding = nn.Embedding(1000, 64)\ntext_projection = nn.Linear(64, 64)\nimages = torch.randn(8, 3, 16, 16)\ntokens = torch.randint(0, 1000, (8, 6))\nzi = F.normalize(image_encoder(images), dim=-1)\nzt = F.normalize(text_projection(text_embedding(tokens).mean(1)), dim=-1)\nlogits = zi @ zt.T / 0.07\nlabels = torch.arange(images.size(0))\nloss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2\nprint('CLIP-style similarity matrix/loss:', logits.shape, float(loss))","Neural Ordinary Differential Equation (Neural ODE)":"import torch\nfrom torch import nn\n\nclass Dynamics(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(dim, 32), nn.Tanh(), nn.Linear(32, dim))\n    def forward(self, z):\n        return self.net(z)\n\n# Differentiable fixed-step Euler solver; torchdiffeq can replace this for adaptive solvers.\nf = Dynamics(8)\nz = torch.randn(4, 8)\ndt = 0.1\nfor _ in range(10):\n    z = z + dt * f(z)\nprint('Neural ODE Euler-integrated state:', z.shape)"}

for _item in ARCHITECTURES:
    if _item['name'] in RUNNABLE_PATCHES:
        _item['code'] = RUNNABLE_PATCHES[_item['name']]

NEW_ARCHITECTURES = [{"name":"DenseNet","family":"Convolutional / vision","idea":"Concatenates feature maps from preceding layers so later layers can reuse earlier features.","flow":"Input → dense convolutional layers with feature concatenation → transition layer → classifier","strengths":"Feature reuse and strong gradient flow.","limitations":"Concatenation increases activation memory; channel growth needs transition layers.","uses":"Image classification and feature extraction.","shapes":"Image (B,3,H,W) → class logits (B,K).","code":"import torch\nfrom torch import nn\n\nclass DenseLayer(nn.Module):\n    def __init__(self, channels, growth=12):\n        super().__init__()\n        self.net = nn.Sequential(nn.BatchNorm2d(channels), nn.ReLU(), nn.Conv2d(channels, growth, 3, padding=1))\n    def forward(self, x):\n        return torch.cat([x, self.net(x)], dim=1)\n\nx = torch.randn(2, 24, 32, 32)\nblock = DenseLayer(24, growth=12)\ny = block(x)\nprint('DenseNet feature shape:', y.shape)"},{"name":"InceptionNet (GoogLeNet-style)","family":"Convolutional / vision","idea":"Processes a feature map through parallel convolution and pooling branches, then concatenates their outputs.","flow":"Input → parallel 1×1, 3×3, factorized convolution and pooling branches → concatenate","strengths":"Captures multiple receptive-field scales efficiently.","limitations":"Branch widths and tensor shapes must be coordinated; full versions are more elaborate.","uses":"Image classification and multi-scale visual features.","shapes":"Image (B,C,H,W) → multi-scale feature map.","code":"import torch\nfrom torch import nn\n\nclass InceptionBlock(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.b1 = nn.Conv2d(c, 8, 1)\n        self.b3 = nn.Sequential(nn.Conv2d(c, 8, 1), nn.ReLU(), nn.Conv2d(8, 12, 3, padding=1))\n        self.pool = nn.Sequential(nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(c, 8, 1))\n    def forward(self, x):\n        return torch.cat([self.b1(x), self.b3(x), self.pool(x)], dim=1)\n\nx = torch.randn(2, 32, 28, 28)\ny = InceptionBlock(32)(x)\nprint('Inception output:', y.shape)"},{"name":"BERT-style Bidirectional Encoder","family":"Attention / language","idea":"Uses bidirectional self-attention so each token can incorporate left and right context; commonly trained with masked-language objectives.","flow":"Token IDs → embeddings + positions → bidirectional Transformer encoder → token-level/classification head","strengths":"Strong contextual representations for language understanding.","limitations":"Encoder-only models are not naturally autoregressive generators; padding and attention masks matter.","uses":"Text classification, token labeling, retrieval embeddings.","shapes":"Token IDs (B,T) → contextual states (B,T,D).","code":"import torch\nfrom torch import nn\n\nvocab, width, length = 1000, 64, 12\nembedding = nn.Embedding(vocab, width)\nlayer = nn.TransformerEncoderLayer(d_model=width, nhead=4, batch_first=True)\nencoder = nn.TransformerEncoder(layer, num_layers=2)\ntokens = torch.randint(0, vocab, (3, length))\npositions = torch.arange(length).unsqueeze(0)\nh = embedding(tokens) + embedding(positions % vocab)\ncontext = encoder(h)  # bidirectional: no causal mask\nmlm_logits = nn.Linear(width, vocab)(context)\nprint('BERT-style masked-token logits:', mlm_logits.shape)"},{"name":"YOLO-style Single-Stage Detector","family":"Convolutional / detection","idea":"Predicts objectness, bounding-box coordinates, and class scores densely from feature maps in a single forward pass.","flow":"Image → CNN backbone → detection head → grid/anchor predictions → decode + non-max suppression","strengths":"Fast single-stage detection with joint localization and classification.","limitations":"This is a compact educational head, not a full trained YOLO release; decoding and NMS are separate post-processing steps.","uses":"Real-time object detection in images and video.","shapes":"Image (B,3,H,W) → predictions (B,anchors,5+classes,H',W').","code":"import torch\nfrom torch import nn\n\nclass TinyYOLOHead(nn.Module):\n    def __init__(self, classes=3, anchors=2):\n        super().__init__()\n        self.features = nn.Sequential(nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU())\n        self.predict = nn.Conv2d(32, anchors * (5 + classes), 1)\n        self.anchors, self.classes = anchors, classes\n    def forward(self, x):\n        y = self.predict(self.features(x))\n        b, _, h, w = y.shape\n        return y.view(b, self.anchors, 5 + self.classes, h, w)\n\npred = TinyYOLOHead()(torch.randn(2, 3, 128, 128))\nprint('YOLO-style predictions:', pred.shape)"},{"name":"Capsule Network","family":"Specialized neural networks","idea":"Represents entities as vectors (capsules); vector length can encode presence while orientation encodes instantiation parameters.","flow":"Convolutional features → capsule vectors → routing-by-agreement → class capsules","strengths":"Explicit vector representations and part-whole relationships.","limitations":"Dynamic routing can be computationally expensive and is less common in production vision stacks.","uses":"Research on viewpoint-aware image recognition.","shapes":"Feature vectors (B,N,D) → capsule vectors (B,K,C).","code":"import torch\nfrom torch import nn\n\ndef squash(v, dim=-1, eps=1e-8):\n    norm2 = (v * v).sum(dim=dim, keepdim=True)\n    return (norm2 / (1 + norm2)) * v / torch.sqrt(norm2 + eps)\n\nclass CapsuleLayer(nn.Module):\n    def __init__(self, in_caps, in_dim, out_caps, out_dim):\n        super().__init__()\n        self.weight = nn.Parameter(torch.randn(1, in_caps, out_caps, out_dim, in_dim) * 0.05)\n    def forward(self, x):\n        u = torch.matmul(self.weight, x[:, :, None, :, None]).squeeze(-1).squeeze(-1)\n        logits = torch.zeros(x.size(0), x.size(1), u.size(2), device=x.device)\n        for _ in range(3):\n            coupling = logits.softmax(dim=-1)\n            v = squash((coupling[..., None] * u).sum(dim=1))\n            logits = logits + (u * v[:, None]).sum(dim=-1)\n        return v\n\ncaps = CapsuleLayer(8, 16, 4, 8)(torch.randn(2, 8, 16))\nprint('Class capsules:', caps.shape)"}]
existing = {item['name'] for item in ARCHITECTURES}
for item in NEW_ARCHITECTURES:
    if item['name'] not in existing:
        ARCHITECTURES.append(item)
ARCH_BY_NAME = {item['name']: item for item in ARCHITECTURES}
print(f"Architecture catalog: {len(ARCHITECTURES)} entries; every entry has a runnable example.")

In [ ]:
def show_architecture(name, show_code=True):
    """Print a structured architecture card. Works without ipywidgets."""
    item = ARCH_BY_NAME[name]
    print("=" * 88)
    print(item["name"], "|", item["family"])
    print("-" * 88)
    print("CORE IDEA\n", textwrap.fill(item["idea"], 86))
    print("\nDATA FLOW\n", item["flow"])
    print("\nSTRENGTHS\n", textwrap.fill(item["strengths"], 86))
    print("\nLIMITATIONS\n", textwrap.fill(item["limitations"], 86))
    print("\nREPRESENTATIVE USE CASES\n", textwrap.fill(item["uses"], 86))
    print("\nTENSOR SHAPES\n", item["shapes"])
    if show_code:
        print("\nPYTORCH PATTERN\n")
        print(item["code"])


In [ ]:
# Interactive architecture browser
if WIDGETS_AVAILABLE:
    family_names = ["All families"] + sorted({a["family"] for a in ARCHITECTURES})
    family_dropdown = widgets.Dropdown(options=family_names, value="All families",
                                       description="Family:", layout=widgets.Layout(width="60%"))
    architecture_dropdown = widgets.Dropdown(
        options=[a["name"] for a in ARCHITECTURES],
        description="Model:", layout=widgets.Layout(width="80%")
    )
    detail_output = widgets.Output()

    def update_model_options(*_):
        family = family_dropdown.value
        names = [a["name"] for a in ARCHITECTURES
                 if family == "All families" or a["family"] == family]
        architecture_dropdown.options = names
        if names:
            architecture_dropdown.value = names[0]

    def render_selected(*_):
        with detail_output:
            detail_output.clear_output(wait=True)
            show_architecture(architecture_dropdown.value, show_code=True)

    family_dropdown.observe(update_model_options, names="value")
    architecture_dropdown.observe(render_selected, names="value")
    update_model_options()
    display(widgets.VBox([family_dropdown, architecture_dropdown, detail_output]))
    render_selected()
else:
    print("Install ipywidgets for the dropdown browser. Use show_architecture('ResNet (Residual Network)') meanwhile.")


## 3. Architecture family map

A useful mental model is to choose an architecture based on the **structure of the data**:

| Data / problem structure | Strong starting families |
|---|---|
| Fixed-size rows / tabular features | MLP; gradient-boosted trees are also important non-neural baselines |
| Images / spatial grids | CNN, ResNet, EfficientNet, ViT, Swin |
| Pixel-level output / segmentation | U-Net, encoder–decoder CNN/Transformer |
| Ordered sequences / moderate time series | LSTM, GRU, Transformer |
| Very long sequences | efficient attention, SSM/Mamba, hybrid models |
| Graphs / relationships | GCN, GAT, other GNNs |
| Generation | autoregressive Transformer, VAE, GAN, diffusion/DiT |
| Similarity / retrieval | Siamese, contrastive dual encoders |
| Multiple modalities | dual encoders, cross-attention, Perceiver-style models |
| Continuous dynamics / scientific data | Neural ODE, SSM, specialized physics-informed networks |

There is no universally best architecture. Start with a simple baseline, define a reliable evaluation split, and compare models using the same data and metric.


## 4. Hands-on lab: train a tiny neural network from scratch

This lab implements a two-layer MLP using **NumPy only**. It learns a nonlinear two-class pattern. You will see:
- forward propagation
- sigmoid activation
- binary cross-entropy loss
- gradients computed with backpropagation
- gradient-descent weight updates
- a loss curve and learned decision boundary

The goal is to understand the mechanics, not to build a production training framework.


In [ ]:
# Generate a two-moons-style dataset without scikit-learn
def make_two_moons(n=600, noise=0.12, seed=7):
    r = np.random.default_rng(seed)
    n1 = n // 2
    t1 = r.uniform(0, np.pi, n1)
    t2 = r.uniform(0, np.pi, n - n1)
    moon1 = np.c_[np.cos(t1), np.sin(t1)]
    moon2 = np.c_[1 - np.cos(t2), 0.5 - np.sin(t2)]
    X = np.vstack([moon1, moon2]) + r.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(n1), np.ones(n - n1)].reshape(-1, 1)
    order = r.permutation(n)
    return X[order], y[order]

X, y = make_two_moons()
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train[:, 0], s=18)
plt.title("Synthetic nonlinear classification data")
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.grid(alpha=0.2); plt.show()


In [ ]:
# NumPy MLP with manual backpropagation
def train_numpy_mlp(X, y, hidden=16, learning_rate=0.8, epochs=2500, seed=3):
    r = np.random.default_rng(seed)
    n_features = X.shape[1]
    W1 = r.normal(0, np.sqrt(2 / n_features), (n_features, hidden))
    b1 = np.zeros((1, hidden))
    W2 = r.normal(0, np.sqrt(1 / hidden), (hidden, 1))
    b2 = np.zeros((1, 1))
    losses = []

    for epoch in range(epochs):
        # Forward pass
        z1 = X @ W1 + b1
        a1 = np.maximum(0, z1)          # ReLU
        z2 = a1 @ W2 + b2
        y_hat = sigmoid(z2)

        # Binary cross entropy (clip avoids log(0))
        eps = 1e-8
        loss = -np.mean(y*np.log(y_hat+eps) + (1-y)*np.log(1-y_hat+eps))
        losses.append(loss)

        # Backward pass: gradients
        m = len(X)
        dz2 = (y_hat - y) / m
        dW2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)
        da1 = dz2 @ W2.T
        dz1 = da1 * (z1 > 0)
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # Gradient descent update
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    params = (W1, b1, W2, b2)
    return params, np.asarray(losses)

def predict_numpy_mlp(X, params):
    W1, b1, W2, b2 = params
    a1 = np.maximum(0, X @ W1 + b1)
    return sigmoid(a1 @ W2 + b2)

params, losses = train_numpy_mlp(X_train, y_train)
test_prob = predict_numpy_mlp(X_test, params)
test_pred = (test_prob >= 0.5).astype(int)
accuracy = np.mean(test_pred == y_test)

print(f"Test accuracy: {accuracy:.3f}")
print(f"Initial loss: {losses[0]:.4f} | Final loss: {losses[-1]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.title("Training loss")
plt.xlabel("Epoch"); plt.ylabel("Binary cross-entropy")
plt.grid(alpha=0.25); plt.show()


In [ ]:
# Visualize the learned decision boundary
grid_x, grid_y = np.meshgrid(
    np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 220),
    np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 220)
)
grid = np.c_[grid_x.ravel(), grid_y.ravel()]
probs = predict_numpy_mlp(grid, params).reshape(grid_x.shape)

plt.figure(figsize=(7, 5.5))
plt.contourf(grid_x, grid_y, probs, levels=20, alpha=0.65)
plt.contour(grid_x, grid_y, probs, levels=[0.5], linewidths=2)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test[:, 0], edgecolors="black", s=24)
plt.title("MLP decision boundary (test points)")
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.show()


### What to change

Try changing:
- `hidden=16` to `4`, `32`, or `64`
- `learning_rate=0.8` to `0.1` or `1.5`
- `epochs=2500` to `300`
- the activation from ReLU to `np.tanh` (remember to change its derivative in backpropagation)

Observe whether the model underfits, learns slowly, or becomes unstable. Keep the test set separate from training.


## 5. Tensor-shape lab: trace data through common layers

These are typical PyTorch conventions:
- Dense/MLP: `(batch, features)`
- CNN: `(batch, channels, height, width)`
- RNN/LSTM/GRU with `batch_first=True`: `(batch, time, features)`
- Transformer with `batch_first=True`: `(batch, tokens, hidden_dim)`
- GNN: often `(nodes, features)` plus an edge index.

The next cell prints a few shape transformations using NumPy equivalents.


In [ ]:
# Shape reasoning without needing PyTorch
B, C, H, W = 8, 3, 224, 224
patch = 16
n_patches = (H // patch) * (W // patch)
d_model = 192

print(f"Image batch:                 ({B}, {C}, {H}, {W})")
print(f"ViT patch tokens:            ({B}, {n_patches}, {d_model})")
print(f"Sequence batch (T=100,F=12): ({B}, 100, 12)")
print(f"Transformer hidden states:   ({B}, 100, {d_model})")
print(f"Tabular feature batch:       ({B}, 40)")
print(f"Graph node features:         (number_of_nodes, feature_dim)")


## 6. Use-case catalog: 40 representative applications

These are **problem patterns**, not claims that one architecture is always best. The architecture column lists reasonable starting families; validate against simpler baselines and domain constraints.


| # | Use case | Data shape | Candidate architecture families |\n|---:|---|---|---|\n| 1 | Customer churn / retention | Tabular + behavioral sequences | MLP; LSTM/Transformer for histories |\n| 2 | Fraud detection | Tabular + transaction graph | MLP; GNN; sequence Transformer |\n| 3 | Credit risk scoring | Tabular | MLP |\n| 4 | Predictive maintenance | Sensor time series | 1D CNN; LSTM/GRU; SSM |\n| 5 | Energy load forecasting | Time series | LSTM; Transformer; SSM |\n| 6 | Network traffic anomaly detection | Time series + events | Autoencoder; 1D CNN; Transformer |\n| 7 | Image classification | Images | CNN; ResNet; EfficientNet; ViT |\n| 8 | Object detection | Images/video | CNN/ResNet; Vision Transformer detectors |\n| 9 | Semantic segmentation | Images | U-Net; encoder–decoder CNN/Transformer |\n| 10 | Medical image analysis | 2D/3D images | U-Net; CNN; ViT; hybrid CNN-Transformer |\n| 11 | Optical character recognition | Images + text | CNN/ViT encoder + sequence decoder |\n| 12 | Face / product verification | Image pairs | Siamese; contrastive encoder |\n| 13 | Visual search | Image + text | Dual encoder / CLIP-style |\n| 14 | Speech recognition | Audio sequence | CNN + Transformer; Conformer-style |\n| 15 | Text-to-speech | Text + audio | Transformer; diffusion/vocoder components |\n| 16 | Speaker identification | Audio | CNN; Transformer; metric learning |\n| 17 | Machine translation | Text sequences | Encoder–decoder Transformer |\n| 18 | Document classification | Text | Transformer encoder; MLP head |\n| 19 | Semantic search | Text embeddings | Transformer encoder; dual encoder |\n| 20 | Question answering | Text / documents | Encoder–decoder or decoder-only Transformer |\n| 21 | Code completion | Code tokens | Decoder-only Transformer |\n| 22 | Text summarization | Long text | Encoder–decoder or decoder-only Transformer |\n| 23 | Image generation | Images / latent tensors | Diffusion U-Net; DiT; GAN; VAE |\n| 24 | Video generation | Video / spatiotemporal tokens | Diffusion Transformer; 3D U-Net; hybrid |\n| 25 | Anomaly detection | Tabular / image / time series | Autoencoder; VAE; CNN; Transformer |\n| 26 | Recommendation systems | User-item interactions | Embedding models; GNN; Transformer; MoE |\n| 27 | Molecular property prediction | Molecular graphs / sequences | GNN; MLP; Transformer |\n| 28 | Drug discovery support | Molecules / proteins | GNN; geometric networks; Transformer |\n| 29 | Protein structure / function | Protein sequences / 3D structures | Transformer; geometric GNN |\n| 30 | Traffic prediction | Road network + time series | GNN + temporal model |\n| 31 | Autonomous vehicle perception | Camera / lidar / radar | CNN; ViT; BEV Transformer; multimodal fusion |\n| 32 | Robot manipulation | Images + proprioception + actions | CNN/ViT + policy network; diffusion policy |\n| 33 | Reinforcement-learning control | States / actions / rewards | DQN; actor-critic networks; Transformers |\n| 34 | Satellite / remote sensing | Large images + geospatial data | CNN; U-Net; ViT; Mamba variants |\n| 35 | Weather / climate downscaling | Gridded spatiotemporal data | CNN; U-Net; Transformer; neural operator |\n| 36 | Industrial visual inspection | Images / video | CNN; ViT; anomaly autoencoder |\n| 37 | Music/audio generation | Audio tokens / spectrograms | Autoregressive Transformer; diffusion |\n| 38 | 3D reconstruction | Images / point clouds | NeRF; point-cloud networks; 3D CNN |\n| 39 | Knowledge graph completion | Graph triples | GNN; graph embedding networks |\n| 40 | Scientific simulation surrogate | Physical fields / parameters | Neural operator; FNO; DeepONet; Neural ODE |\n

**How to use this catalog:** pick the row that resembles your problem, then inspect the architecture browser. For a real project, define the target, leakage-safe split, metric, baseline, latency budget, and monitoring plan before selecting a large model.


In [ ]:
USE_CASES = [('Customer churn / retention', 'Tabular + behavioral sequences', 'MLP; LSTM/Transformer for histories', 'Predict probability of churn from usage, tenure, support, and billing signals.'), ('Fraud detection', 'Tabular + transaction graph', 'MLP; GNN; sequence Transformer', 'Flag suspicious transactions, accounts, or coordinated behavior.'), ('Credit risk scoring', 'Tabular', 'MLP', 'Estimate default risk with calibration, fairness, and explainability checks.'), ('Predictive maintenance', 'Sensor time series', '1D CNN; LSTM/GRU; SSM', 'Forecast failures from vibration, temperature, and telemetry.'), ('Energy load forecasting', 'Time series', 'LSTM; Transformer; SSM', 'Forecast demand and detect unusual consumption patterns.'), ('Network traffic anomaly detection', 'Time series + events', 'Autoencoder; 1D CNN; Transformer', 'Identify abnormal traffic patterns or service behavior.'), ('Image classification', 'Images', 'CNN; ResNet; EfficientNet; ViT', 'Assign labels to images, products, defects, or scenes.'), ('Object detection', 'Images/video', 'CNN/ResNet; Vision Transformer detectors', 'Locate and classify multiple objects in a frame.'), ('Semantic segmentation', 'Images', 'U-Net; encoder–decoder CNN/Transformer', 'Assign a class to every pixel, such as road, tumor, or crop.'), ('Medical image analysis', '2D/3D images', 'U-Net; CNN; ViT; hybrid CNN-Transformer', 'Assist image triage, segmentation, and measurement; requires clinical validation.'), ('Optical character recognition', 'Images + text', 'CNN/ViT encoder + sequence decoder', 'Convert scanned documents and images into text.'), ('Face / product verification', 'Image pairs', 'Siamese; contrastive encoder', 'Determine whether two images depict the same identity or item.'), ('Visual search', 'Image + text', 'Dual encoder / CLIP-style', 'Retrieve visually or semantically similar items.'), ('Speech recognition', 'Audio sequence', 'CNN + Transformer; Conformer-style', 'Convert audio to token sequences.'), ('Text-to-speech', 'Text + audio', 'Transformer; diffusion/vocoder components', 'Generate speech conditioned on text.'), ('Speaker identification', 'Audio', 'CNN; Transformer; metric learning', 'Recognize or cluster speakers from voice embeddings.'), ('Machine translation', 'Text sequences', 'Encoder–decoder Transformer', 'Translate between languages.'), ('Document classification', 'Text', 'Transformer encoder; MLP head', 'Classify contracts, tickets, reports, and correspondence.'), ('Semantic search', 'Text embeddings', 'Transformer encoder; dual encoder', 'Retrieve passages by meaning rather than exact keywords.'), ('Question answering', 'Text / documents', 'Encoder–decoder or decoder-only Transformer', 'Answer questions grounded in supplied context.'), ('Code completion', 'Code tokens', 'Decoder-only Transformer', 'Predict likely next tokens or code blocks.'), ('Text summarization', 'Long text', 'Encoder–decoder or decoder-only Transformer', 'Generate concise summaries while preserving key information.'), ('Image generation', 'Images / latent tensors', 'Diffusion U-Net; DiT; GAN; VAE', 'Generate images from prompts or latent inputs.'), ('Video generation', 'Video / spatiotemporal tokens', 'Diffusion Transformer; 3D U-Net; hybrid', 'Generate or extend video clips.'), ('Anomaly detection', 'Tabular / image / time series', 'Autoencoder; VAE; CNN; Transformer', 'Identify examples that differ from learned normal patterns.'), ('Recommendation systems', 'User-item interactions', 'Embedding models; GNN; Transformer; MoE', 'Rank products, media, or content for a user.'), ('Molecular property prediction', 'Molecular graphs / sequences', 'GNN; MLP; Transformer', 'Predict molecular properties from structure or sequence.'), ('Drug discovery support', 'Molecules / proteins', 'GNN; geometric networks; Transformer', 'Model molecular interactions and prioritize candidates; experimental validation required.'), ('Protein structure / function', 'Protein sequences / 3D structures', 'Transformer; geometric GNN', 'Learn sequence and structural representations.'), ('Traffic prediction', 'Road network + time series', 'GNN + temporal model', 'Forecast speed, congestion, or travel-time patterns.'), ('Autonomous vehicle perception', 'Camera / lidar / radar', 'CNN; ViT; BEV Transformer; multimodal fusion', 'Detect objects, lanes, occupancy, and scene structure.'), ('Robot manipulation', 'Images + proprioception + actions', 'CNN/ViT + policy network; diffusion policy', 'Map observations and goals to robot actions.'), ('Reinforcement-learning control', 'States / actions / rewards', 'DQN; actor-critic networks; Transformers', 'Learn policies for control and sequential decision-making.'), ('Satellite / remote sensing', 'Large images + geospatial data', 'CNN; U-Net; ViT; Mamba variants', 'Map land cover, detect changes, and identify objects.'), ('Weather / climate downscaling', 'Gridded spatiotemporal data', 'CNN; U-Net; Transformer; neural operator', 'Learn high-resolution fields from coarse inputs and simulations.'), ('Industrial visual inspection', 'Images / video', 'CNN; ViT; anomaly autoencoder', 'Detect surface defects, missing components, or process deviations.'), ('Music/audio generation', 'Audio tokens / spectrograms', 'Autoregressive Transformer; diffusion', 'Generate, continue, or transform audio.'), ('3D reconstruction', 'Images / point clouds', 'NeRF; point-cloud networks; 3D CNN', 'Recover geometry or view-consistent scene representations.'), ('Knowledge graph completion', 'Graph triples', 'GNN; graph embedding networks', 'Predict missing links or relations in a graph.'), ('Scientific simulation surrogate', 'Physical fields / parameters', 'Neural operator; FNO; DeepONet; Neural ODE', 'Approximate expensive physical simulations for repeated queries.')]


def search_use_cases(term):
    term = term.lower().strip()
    matches = [(i+1, row) for i, row in enumerate(USE_CASES)
               if term in " ".join(row).lower()]
    if not matches:
        print("No matches. Try a broader term such as image, sequence, graph, or audio.")
    for idx, (name, data, arch, desc) in matches:
        print(f"{idx}. {name} | Data: {data} | Models: {arch}\\n   {desc}\\n")
    return matches

if WIDGETS_AVAILABLE:
    use_case_search = widgets.Text(value="", placeholder="e.g. image, fraud, time series, audio",
                                   description="Search:", layout=widgets.Layout(width="80%"))
    use_case_output = widgets.Output()
    def run_use_case_search(change=None):
        with use_case_output:
            use_case_output.clear_output(wait=True)
            search_use_cases(use_case_search.value)
    use_case_search.observe(run_use_case_search, names="value")
    display(widgets.VBox([use_case_search, use_case_output]))
    run_use_case_search()


## 7. Production engineering: model architecture vs. system architecture

A neural architecture describes the learnable computation graph. A production ML system also needs data validation, training orchestration, evaluation, model packaging, serving, monitoring, and rollback.

A practical pipeline might look like:

`ingest → validate → preprocess/features → train → evaluate → register → deploy → monitor`

For distributed training, TensorFlow offers strategies such as `MirroredStrategy` (multiple GPUs on one machine) and `MultiWorkerMirroredStrategy` (multiple workers). Kubernetes schedules containers and resources; it does not itself make a model train in parallel. The training framework and distributed strategy coordinate the computation.

### TensorFlow distributed training sketch

```python
import tensorflow as tf

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(10)
    ])
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )
```

This code uses multiple compatible GPUs on one machine when available. For multi-worker training, investigate `MultiWorkerMirroredStrategy`; for Kubernetes-managed jobs, investigate TensorFlowJob/Kubeflow training operators and current framework guidance.

### Distinguish two scaling problems

| Concern | Distributed training | Distributed inference |
|---|---|---|
| Goal | Train model parameters faster or across more data | Serve more prediction requests |
| Main resources | GPUs/accelerators, network, training data | CPU/GPU, memory, request concurrency |
| Coordination | Gradient synchronization / parameter updates | Load balancing, batching, replicas |
| Metrics | Throughput, step time, scaling efficiency | Latency percentiles, throughput, error rate |
| Risks | Stragglers, communication overhead, divergence | Cold starts, queueing, model loading, saturation |


## 8. Newer and research-frontier directions

These are worth knowing, but treat them as active research areas rather than automatic replacements for established architectures:

- **Selective State Space Models / Mamba:** input-dependent state updates for efficient long-sequence processing.
- **Vision Mamba and hybrid CNN/Transformer/SSM models:** adapt sequence/state-space ideas to images and video.
- **Mixture-of-Experts (MoE):** sparse conditional computation, often used to grow model capacity.
- **Diffusion Transformers (DiT):** Transformer blocks as denoising backbones.
- **Neural operators (FNO, DeepONet):** learn mappings between functions/fields, useful in scientific computing.
- **KANs:** learnable functions on edges; promising for some interpretable approximation tasks, but empirical trade-offs remain under study.
- **Multimodal foundation models:** combine text, images, audio, video, and sometimes action or spatial tokens.

A good research habit is to ask: What is the architectural novelty? What is the compute/memory cost? Which benchmark and data regime support the claim? Is there a reproducible implementation? Does it beat a strong baseline?


## 9. Mini assessment

Answer these before moving on:

1. Why is a Transformer not simply an MLP with attention added?
2. What does a residual connection change about information and gradient flow?
3. Why can a CNN be a better first model than a ViT for a small image dataset?
4. What is the difference between an LSTM cell state and hidden state?
5. Why does full self-attention become expensive as sequence length grows?
6. What does an MoE router decide?
7. How does a GNN use edge connectivity?
8. Why is backpropagation a training mechanism rather than an architecture?
9. What is the difference between distributed training and replicated inference?
10. What baseline would you build before choosing a bleeding-edge architecture?

**Suggested answers:** explain each in your own words, then use `show_architecture(...)` to inspect the relevant model and tensor shapes.


## 10. Reference material and further reading

### User-provided starting links
- [Google search: neural network architectures](https://www.google.com/search?q=neural+network+architectures) — discovery link supplied by the user; not treated as a single article.
- [Real Python: Python AI / neural network learning resource](https://realpython.com/quizzes/python-ai-neural-network/viewer/)
- [Google Machine Learning Crash Course: Neural Networks](https://developers.google.com/machine-learning/crash-course/neural-networks)
- [Optibrium: Machine Learning 101 — How to Build Your First Neural Network](https://optibrium.com/knowledge-base/machine-learning-101-how-to-build-your-first-neural-network/)
- [Matroid: Scaling Convolutional Neural Networks with Kubernetes and TensorFlow](https://www.matroid.com/scaling-convolutional-neural-networks-with-kubernetes-and-tensorflow/)

### Additional technical references
- [PyTorch tutorials](https://pytorch.org/tutorials/)
- [TensorFlow distributed training guide](https://www.tensorflow.org/guide/distributed_training)
- [PyTorch Geometric documentation](https://pytorch-geometric.readthedocs.io/)
- [A survey on deep learning fundamentals (2025)](https://link.springer.com/article/10.1007/s10462-025-11368-7)
- [A survey of Mamba (2026)](https://doi.org/10.1145/3819582)
- [Revisiting U-Net: a foundational backbone for modern generative AI](https://link.springer.com/article/10.1007/s10462-025-11450-0)

The notebook is a learning guide, not a claim that every architecture is state of the art on every task. Check original papers, maintained implementations, and current benchmarks before making production choices.


---
### Suggested next progression

1. Work through the NumPy MLP and explain every gradient.
2. Rebuild the same model in PyTorch and inspect `loss.backward()`, `.grad`, and the optimizer step.
3. Implement a CNN and visualize feature-map shapes.
4. Build a small attention operation from matrix multiplications before using `nn.MultiheadAttention`.
5. Compare LSTM, Transformer, and a selective SSM conceptually on a long synthetic sequence.
6. Turn one use case into an end-to-end training/evaluation/serving mini-project.

**Keep the notebook practical:** change one variable at a time, run the cell, inspect shapes and plots, and record what changed.


## 11. Classical ML catalog — runnable Python examples

These examples use scikit-learn and small synthetic datasets. Run each cell independently.

### Linear / Ridge / Lasso Regression

In [ ]:
from sklearn.linear_model import Ridge
X=[[0,1],[1,0],[2,1],[3,0]]; y=[1,2,3,4]
m=Ridge(alpha=1.0).fit(X,y)
print('prediction:', m.predict([[1.5,0.5]]))

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
X=[[0],[1],[2],[3],[4],[5]]; y=[0,0,0,1,1,1]
m=LogisticRegression().fit(X,y)
print('probability:', m.predict_proba([[2.5]]))

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
X=[[0,0],[0,1],[1,0],[1,1]]; y=[0,0,0,1]
m=DecisionTreeClassifier(max_depth=2,random_state=42).fit(X,y)
print('class:', m.predict([[1,1]]))

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
X=[[0],[1],[2],[3],[4],[5]]; y=[0,0,0,1,1,1]
m=RandomForestClassifier(n_estimators=30,random_state=42).fit(X,y)
print('class:', m.predict([[4.2]]))

### Gradient Boosting (XGBoost / LightGBM / CatBoost family)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
X=[[0],[1],[2],[3],[4],[5]]; y=[0,0,0,1,1,1]
m=HistGradientBoostingClassifier(random_state=42).fit(X,y)
print('class:', m.predict([[4.5]]))

### Support Vector Machine

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
X=[[-2],[-1],[1],[2]]; y=[0,0,1,1]
m=make_pipeline(StandardScaler(),SVC(probability=True)).fit(X,y)
print('class:',m.predict([[1.5]]))

### k-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
X=[[0,0],[0,1],[1,0],[5,5],[5,6],[6,5]]; y=[0,0,0,1,1,1]
m=KNeighborsClassifier(n_neighbors=3).fit(X,y)
print('class:',m.predict([[5.2,5.1]]))

### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
X=[[1,2],[1,3],[4,5],[5,4]]; y=[0,0,1,1]
m=GaussianNB().fit(X,y)
print('class:',m.predict([[4,4]]))

### k-Means

In [ ]:
from sklearn.cluster import KMeans
X=[[0,0],[0,1],[1,0],[8,8],[8,9],[9,8]]
m=KMeans(n_clusters=2,n_init=10,random_state=42).fit(X)
print('cluster labels:',m.labels_)

### DBSCAN / HDBSCAN family

In [ ]:
from sklearn.cluster import DBSCAN
X=[[0,0],[0,.1],[.1,0],[5,5],[5.1,5],[20,20]]
labels=DBSCAN(eps=.3,min_samples=2).fit_predict(X)
print('cluster labels (-1=noise):',labels)

### Principal Component Analysis (PCA)

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
X=np.array([[1,2,3],[2,3,4],[3,4,5],[4,5,6]],dtype=float)
z=PCA(n_components=2).fit_transform(X)
print('reduced shape:',z.shape)

### Genetic Algorithm

In [ ]:
import random
random.seed(7)
pop=[random.uniform(-5,5) for _ in range(20)]
fitness=lambda x: -(x-2.0)**2
for _ in range(30):
    elite=sorted(pop,key=fitness,reverse=True)[:5]
    pop=elite+[max(-5,min(5,x+random.gauss(0,.35))) for x in elite for _ in range(3)]
print('best solution:',max(pop,key=fitness))

## 12. Artificial Intelligence catalog — runnable Python examples

These examples cover search, symbolic reasoning, retrieval, language-model inference, agents, and planning. The LLM example downloads a small Hugging Face checkpoint and requires `transformers` plus internet access on first run.

### Search (BFS / DFS / A*)

In [ ]:
from collections import deque
graph={'A':['B','C'],'B':['D'],'C':['E'],'D':[],'E':[]}
q=deque(['A']); seen=set(); order=[]
while q:
 n=q.popleft()
 if n not in seen:
  seen.add(n); order.append(n); q.extend(graph[n])
print('BFS:',order)

### Minimax / Alpha-Beta Pruning

In [ ]:
def minimax(tree, maximizing=True):
 if isinstance(tree,int): return tree
 vals=[minimax(x,not maximizing) for x in tree]
 return max(vals) if maximizing else min(vals)
print('best value:',minimax([[3,5],[2,9]]))

### Monte Carlo Tree Search (rollout core)

In [ ]:
import random
random.seed(3)
# Estimate the value of two candidate actions by random rollouts.
rewards={0:[0,1,1,0,1],1:[0,0,1,0,0]}
means={a:sum(rewards[a])/len(rewards[a]) for a in rewards}
print('estimated action values:',means,'choose:',max(means,key=means.get))

### Rule Engines / Expert Systems

In [ ]:
rules=[(lambda f:f['temperature']>38,'possible fever'),(lambda f:f['battery']<20,'low battery')]
facts={'temperature':39,'battery':12}
print([message for test,message in rules if test(facts)])

### Bayesian Networks

In [ ]:
# Tiny Bayesian update: P(H|E) = P(E|H)P(H) / P(E)
p_h=.01; p_e_given_h=.9; p_e_given_not_h=.05
p_e=p_e_given_h*p_h+(p_e_given_not_h*(1-p_h))
print('posterior:',p_e_given_h*p_h/p_e)

### Constraint Satisfaction / Backtracking

In [ ]:
variables=['A','B','C']; domains={v:[1,2,3] for v in variables}
def solve(i,assignment):
 if i==len(variables): return assignment
 v=variables[i]
 for value in domains[v]:
  if all(value!=assignment.get(n) for n in assignment):
   result=solve(i+1,{**assignment,v:value})
   if result:return result
print('coloring:',solve(0,{}))

### Knowledge Graph

In [ ]:
kg={'Alice':{'works_at':['Acme'],'knows':['Bob']},'Bob':{'works_at':['Beta']}}
print('Alice works at:',kg['Alice']['works_at'])
print('Alice knows:',kg['Alice']['knows'])

### Retrieval-Augmented Generation (RAG)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs=['Kubernetes schedules containers.','Terraform provisions infrastructure.','PyTorch trains neural networks.']
q='How do I schedule containers?'
v=TfidfVectorizer().fit(docs+[q]); scores=cosine_similarity(v.transform([q]),v.transform(docs))[0]
i=scores.argmax(); print('retrieved:',docs[i],'score:',round(scores[i],3))

### Large Language Model inference (local checkpoint)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id='distilgpt2'  # downloads a small public checkpoint on first run
tok=AutoTokenizer.from_pretrained(model_id)
model=AutoModelForCausalLM.from_pretrained(model_id)
inputs=tok('Machine learning is',return_tensors='pt')
ids=model.generate(**inputs,max_new_tokens=20)
print(tok.decode(ids[0],skip_special_tokens=True))

### Tool-Using AI Agent

In [ ]:
def get_weather(city): return f'Weather lookup tool called for {city}'
tools={'weather':get_weather}
plan={'tool':'weather','args':{'city':'Dallas'}}
result=tools[plan['tool']](**plan['args'])
print('agent observation:',result)

### Multi-Agent System

In [ ]:
def researcher(task): return {'evidence':f'Collected evidence for {task}'}
def analyst(evidence): return {'assessment':f"Analyzed: {evidence['evidence']}"}
def reviewer(assessment): return 'Approved: '+assessment['assessment']
state=researcher('model latency'); state=analyst(state); print(reviewer(state))

### Planning / Workflow Orchestration

In [ ]:
steps={'validate':['train'],'train':['evaluate'],'evaluate':['package'],'package':[]}
order=[]; ready=['validate']
while ready:
 step=ready.pop(0); order.append(step); ready.extend(steps[step])
print('workflow order:',order)